### Global Imports

In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

### Global Constants

**SI units throughout**

The standard gravitational parameter $\mu$ is a physical constant that represents the strength of a celestial body's gravitational pull, defined as $\mu = GM$, where $G$ is the universal gravitational constant and $M$ is the mass of the celestial body.


In [34]:
G = 6.674_30e-11
M_EARTH = 5.972_20e+24
R_EARTH = 6.371_00e+6
MU_EARTH = G * M_EARTH

### Orbital Motion

For a satellite moving at constant speed along a circular path of orbital period $T$, the angular speed $\omega$ is constant and given by $\omega = \frac{2\pi}{T}$. The gravity of the celestial body provides the centripetal force required to keep the satellite on its circular path, so $\frac{GMm}{r^2} = m\omega^2 r$. Solving for $\omega$ gives $\omega = \sqrt{\dfrac{\mu}{r^3}}$.

The equations of motion on the plane ($x_3=0$):
$$
\mathbf{x}(t) = r \begin{bmatrix}\cos(\omega t)\\ \sin(\omega t)\\ 0\end{bmatrix}, \quad
\mathbf{v}(t) = r\omega \begin{bmatrix}-\sin(\omega t)\\ \cos(\omega t)\\ 0\end{bmatrix}, \quad
\mathbf{a}(t) = -\omega^2 \mathbf{x}(t).
$$

In [35]:
def angular_speed(r: float, mu: float) -> float:
    return np.sqrt(mu/r**3)

def position(t: np.ndarray, r: float, omega: float) -> np.ndarray:
    x1 = r * np.cos(omega * t)
    x2 = r * np.sin(omega * t)
    x3 = np.zeros_like(t)

    return np.stack((x1, x2, x3), axis=-1)

def velocity(t: np.ndarray, r: float, omega: float) -> np.ndarray:
    v1 = -r * omega * np.sin(omega * t)
    v2 =  r * omega * np.cos(omega * t)
    v3 = np.zeros_like(t)

    return np.stack((v1, v2, v3), axis=-1)

def acceleration(x: np.ndarray, omega: float) -> np.ndarray:
    return -omega**2 * x


### Rotational System

Our circular path lives on a 2D-plane, which can be rotated to give configurations in 3D-geometric space. The rotational matrices are:
$$
\mathbf{R_x}(\alpha) = \begin{bmatrix}
    1 & 0 & 0 \\ 
    0 & \cos(\alpha) & -\sin(\alpha) \\ 
    0 & \sin(\alpha) &  \cos(\alpha)
\end{bmatrix}, \quad

\mathbf{R_y}(\beta)  = \begin{bmatrix}
     \cos(\beta) & 0 & \sin(\beta) \\ 
    0 & 1 & 0 \\ 
    -\sin(\beta) & 0 & \cos(\beta)
\end{bmatrix}, \quad

\mathbf{R_z}(\gamma) = \begin{bmatrix}
    \cos(\gamma) & -\sin(\gamma) & 0 \\ 
    \sin(\gamma) &  \cos(\gamma) & 0 \\ 
    0 & 0 & 1
\end{bmatrix}.
$$

Any two combinations of the rotation matrices $\mathbf{x'} = \mathbf{R_x}(\alpha) \mathbf{R_z}(\gamma) \mathbf{x}$, permits all orientations of the plane.



In [36]:
def rotation_x(alpha: float) -> np.ndarray:
    return np.array([
        [1, 0, 0],
        [0, np.cos(alpha), -np.sin(alpha)],
        [0, np.sin(alpha),  np.cos(alpha)]
    ])

def rotation_y(beta: float) -> np.ndarray:
    return np.array([
        [ np.cos(beta), 0, np.sin(beta)],
        [ 0, 1, 0],
        [-np.sin(beta), 0, np.cos(beta)]
    ])

def rotation_z(gamma: float) -> np.ndarray:
    return np.array([
        [np.cos(gamma), -np.sin(gamma), 0],
        [np.sin(gamma),  np.cos(gamma), 0],
        [0, 0, 1]
    ])


### Some Orbits

- International Space Station Orbit: 400km, 51.6° inclination
- Polar Orbit: 600km, 90° inclination such that it passes over both poles every orbit
- Sun Synchronous Orbit: 600km, ~98.2° slightly retrograde, the near-polar
  orbit used by most Earth-observation satellites
- Geostationary Orbit: 35,786km, 0° inclination equatorial, period matches Earth's rotation
- Retrograde Equatorial Orbit: same altitude as ISS but orbiting backwards
  (inclination = 180°)



In [37]:
@dataclass(frozen=True, slots=True)
class CircularOrbit:
    name: str
    radius: float
    inclination: float
    raan: float

    @property
    def omega(self) -> float:
        return angular_speed(self.radius, MU_EARTH)

    @property
    def period(self) -> float:
        return 2 * np.pi / self.omega

    @property
    def rotation(self) -> np.ndarray:
        return (rotation_z(self.raan) @ rotation_x(self.inclination))
    
    def positions(self, t: np.ndarray) -> np.ndarray:
        x = position(t, self.radius, self.omega)
        return x @ self.rotation.T

    def velocities(self, t: np.ndarray) -> np.ndarray:
        v = velocity(t, self.radius, self.omega)
        return v @ self.rotation.T


orbits = [
    CircularOrbit("geostationary", R_EARTH + 35_786e3, np.deg2rad(0.0), np.deg2rad(0.0)),
    CircularOrbit("iss", R_EARTH + 400e3, np.deg2rad(51.6), np.deg2rad(0.0)),
    CircularOrbit("polar", R_EARTH + 600e3, np.deg2rad(90.0), np.deg2rad(0.0)),
    CircularOrbit("retrograde-equatorial", R_EARTH + 400e3, np.deg2rad(180.0), np.deg2rad(0.0)),
    CircularOrbit("sun-synchronous", R_EARTH + 600e3, np.deg2rad(98.2), np.deg2rad(0.0)),
]


### Evolution Analysis

Show orbital evolution over a 24-hour period, sampled at one-minute intervals. This can be used to obtain valid initial conditions for numerical orbit integration and to compare the resulting trajectories.


In [38]:
DAY_IN_SECONDS = 24 * 60 * 60
t = np.arange(0, DAY_IN_SECONDS + 1, step=60)

rows = []
for orbit in orbits:
    x = orbit.positions(t)
    v = orbit.velocities(t)
    a = acceleration(x, orbit.omega)

    for time, pos, vel, acc in zip(t, x, v, a):
        rows.append({
            "orbit": orbit.name,
            "timedelta": pd.to_timedelta(time, unit="s"), # type: ignore
            "x":  pos[0],  "y": pos[1],  "z": pos[2],
            "vx": vel[0], "vy": vel[1], "vz": vel[2],
            "ax": acc[0], "ay": acc[1], "az": acc[2]
        })

df = pd.DataFrame(rows).set_index(["timedelta", "orbit"]).sort_index()
df.to_csv("output/orbits.csv")

df.loc[pd.to_timedelta(range(0, 25, 6), unit="h")]


x             y  \
timedelta       orbit                                               
0 days 00:00:00 geostationary          4.215700e+07  0.000000e+00   
                iss                    6.771000e+06  0.000000e+00   
                polar                  6.971000e+06  0.000000e+00   
                retrograde-equatorial  6.771000e+06  0.000000e+00   
                sun-synchronous        6.971000e+06  0.000000e+00   
0 days 06:00:00 geostationary         -1.984190e+05  4.215653e+07   
                iss                    5.363497e+06 -2.567027e+06   
                polar                 -9.138137e+05 -4.231672e-10   
                retrograde-equatorial  5.363497e+06  4.132716e+06   
                sun-synchronous       -9.138137e+05  9.856865e+05   
0 days 12:00:00 geostationary         -4.215513e+07 -3.968336e+05   
                iss                    1.726150e+06 -4.066827e+06   
                polar                 -6.731420e+06  1.109442e-10   
                retrograde-equatorial  1.726150e+06  6.547278e+06   
                sun-synchronous       -6.731420e+06 -2.584231e+05   
0 days 18:00:00 geostationary          5.952394e+05 -4.215280e+07   
                iss                   -2.628835e+06 -3.875866e+06   
                polar                  2.678629e+06  3.940804e-10   
                retrograde-equatorial -2.628835e+06  6.239845e+06   
                sun-synchronous        2.678629e+06 -9.179342e+05   
1 days 00:00:00 geostationary          4.214953e+07  7.936321e+05   
                iss                   -5.890896e+06 -2.073535e+06   
                polar                  6.029148e+06 -2.142625e-10   
                retrograde-equatorial -5.890896e+06  3.338232e+06   
                sun-synchronous        6.029148e+06  4.990833e+05   

                                                  z           vx  \
timedelta       orbit                                              
0 days 00:00:00 geostationary          0.000000e+00     0.000000   
                iss                    0.000000e+00     0.000000   
                polar                  0.000000e+00     0.000000   
                retrograde-equatorial  0.000000e+00     0.000000   
                sun-synchronous        0.000000e+00     0.000000   
0 days 06:00:00 geostationary          0.000000e+00 -3074.895593   
                iss                   -3.238782e+06  4683.023707   
                polar                 -6.910845e+06  7496.500825   
                retrograde-equatorial -5.061117e-10  4683.023707   
                sun-synchronous       -6.840191e+06  7496.500825   
0 days 12:00:00 geostationary          0.000000e+00    28.945026   
                iss                   -5.131059e+06  7419.106176   
                polar                  1.811856e+06 -1965.401014   
                retrograde-equatorial -8.018103e-10  7419.106176   
                sun-synchronous        1.793332e+06 -1965.401014   
0 days 18:00:00 geostationary          0.000000e+00  3074.623124   
                iss                   -4.890126e+06  7070.736232   
                polar                  6.435821e+06 -6981.220265   
                retrograde-equatorial -7.641607e-10  7070.736232   
                sun-synchronous        6.370022e+06 -6981.220265   
1 days 00:00:00 geostationary          0.000000e+00   -57.887488   
                iss                   -2.616150e+06  3782.746757   
                polar                 -3.499173e+06  3795.707941   
                retrograde-equatorial -4.088155e-10  3782.746757   
                sun-synchronous       -3.463398e+06  3795.707941   

                                                 vy            vz        ax  \
timedelta       orbit                                                         
0 days 00:00:00 geostationary          3.074930e+03  0.000000e+00 -0.224285   
                iss                    4.765830e+03  6.012981e+03 -8.694296   
                polar            

### Render Orbits in 3D Plot

In [ ]:
NUMBER_OF_POINTS = 1_000_000
PLOT_ZOOM = 0.15

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection="3d")
ax.set_proj_type("ortho")

# mock surface of Earth
u, v = np.mgrid[0:2*np.pi:60j, 0:np.pi:30j]
ex = R_EARTH * np.cos(u) * np.sin(v)
ey = R_EARTH * np.sin(u) * np.sin(v)
ez = R_EARTH * np.cos(v)
ax.plot_surface(ex, ey, ez, color="steelblue",
    alpha=0.4, linewidth=0, shade=True
)

# render orbitals
max_radius = 0.0
for orbit in orbits:
    max_radius = max(max_radius, orbit.radius)
    t = np.linspace(0, 1000 * orbit.period, NUMBER_OF_POINTS)

    p = orbit.positions(t)
    ax.plot(p[:, 0], p[:, 1], p[:, 2], lw=1, label=orbit.name)


lim = max_radius * PLOT_ZOOM
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_zlim(-lim, lim)

ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_zlabel("z (m)")

ax.set_box_aspect([1, 1, 1])
ax.legend(loc="upper left", fontsize=9)

plt.tight_layout()
plt.show()
